In [ ]:
import sys
sys.path.append("../")
from qmpsqsc.models import mpsqsc
from qmpsqsc.models import qmps
from qmpsqsc.models import data as mpsdata

from importlib import reload
reload(mpsdata)

<module 'qmpsqsc.models.data' from '/Users/keisuke/Documents/projects/mps4qsc/notebooks/../qmpsqsc/models/data/__init__.py'>

In [ ]:
import torch.nn.functional as F
import torch

L = 30
chi = 2
d = 2
ghz = mpsqsc.build_ghz_state(L, d, chi).to(dtype = torch.complex128)
ghz = ghz.normalize()

ghz2 = ghz.copy()
ghz2.As[0][:, 1] = -ghz2.As[0][:, 1]

ghz_errors = [mpsdata.flip_sites_in_mps(ghz, [i]) for i in range(L)]
for ghz_error in ghz_errors:
    ghz_error.normalize()

ghz2_errors = [mpsdata.flip_sites_in_mps(ghz2, [i]) for i in range(L)]
for ghz2_error in ghz2_errors:
    ghz2_error.normalize()

ghzs1 = mpsqsc.add_mpstates([ghz] + ghz_errors)
ghzs2 = mpsqsc.add_mpstates([ghz2] + ghz2_errors)
ghzs1 = ghzs1.normalize()
ghzs2 = ghzs2.normalize()


allup = mpsqsc.build_classical_state(L, d, [0]*L).to(dtype=torch.complex128)
alldown = mpsqsc.build_classical_state(L, d, [1]*L).to(dtype=torch.complex128)

allup_errors = [mpsdata.flip_sites_in_mps(allup, [i]) for i in range(L)]
alldown_errors = [mpsdata.flip_sites_in_mps(alldown, [i]) for i in range(L)]

mixed_states = mpsqsc.add_mpstates([allup, alldown] + allup_errors + alldown_errors)
mixed_states = mixed_states.normalize()

In [3]:
data_generator = mpsdata.ghz.create_ghz_rho_batch_qsc(ghz, allup, alldown, 2**5, 0.5)

In [4]:
ghz_qsc = mpsqsc.MpsQsc.load_model("data/ghz_qsc_error.pt")

In [5]:
ghz_qsc.canonicalize(inplace=True, truncate=True, normalize=True)
Us, last = qmps.construct_unitary_from_As(ghz_qsc.As)
chi = 2
qmps_ghz = qmps.qMPS(L, chi, d, Us=Us, last_unitary=last)


In [6]:
states, labels, errors = next(data_generator)
# qmps.predict(states)

In [7]:
from qmpsqsc.models.qmps.optimizer import StiefelAdam
qmps_ghz = qmps.qMPS.load_from("data/qmps_ghz_error_w=1.pt").to(dtype=torch.complex128)

In [8]:
qmps_ghz.set_requires_grad(True)

In [9]:
w = 1
qmps_ghz.set_weights(w)
loss, acc, probs = mpsdata.loss.calculate_loss_qmps(qmps_ghz, states, labels)
print(loss, acc)

tensor(0.3853, dtype=torch.float64, grad_fn=<NegBackward0>) tensor(0.6875)


In [ ]:
# qmps_ghz.save_to("data/qmps_ghz_error_w=1.pt")

In [10]:
optim = StiefelAdam(qmps_ghz.unitaries(), lr=0.01)

for step in range(100):
    optim.zero_grad(set_to_none=True)

    # Shuffle the 4 examples each step
    loss, acc, probs = mpsdata.loss.calculate_loss_qmps(qmps_ghz, states, labels)
    loss.backward()
    optim.step()

    print(f"[step {step:5d}] loss={loss.item():.6f}  acc={acc:.3f}")

[step     0] loss=0.385335  acc=0.688
[step     1] loss=0.385050  acc=0.750
[step     2] loss=0.381832  acc=0.750
[step     3] loss=0.380454  acc=0.750
[step     4] loss=0.378772  acc=0.969
[step     5] loss=0.376438  acc=0.969
[step     6] loss=0.374259  acc=0.969
[step     7] loss=0.372590  acc=0.969
[step     8] loss=0.371165  acc=0.969
[step     9] loss=0.369632  acc=0.969
[step    10] loss=0.368028  acc=0.969
[step    11] loss=0.366578  acc=0.969
[step    12] loss=0.365372  acc=0.969
[step    13] loss=0.364302  acc=0.969
[step    14] loss=0.363241  acc=0.969
[step    15] loss=0.362170  acc=0.969
[step    16] loss=0.361139  acc=0.969
[step    17] loss=0.360193  acc=0.969
[step    18] loss=0.359346  acc=0.969
[step    19] loss=0.358579  acc=0.969
[step    20] loss=0.357850  acc=1.000
[step    21] loss=0.357141  acc=1.000
[step    22] loss=0.356475  acc=1.000
[step    23] loss=0.355875  acc=1.000
[step    24] loss=0.355332  acc=1.000
[step    25] loss=0.354811  acc=1.000
[step    26]

In [27]:
import torch
ghz = ghz.to(torch.complex128)
allup = allup.to(torch.complex128)
alldown = alldown.to(torch.complex128)

AttributeError: 'MPState' object has no attribute 'to'

In [31]:
reload(mpsdata.ghz)
data_generator = mpsdata.ghz.create_ghz_rho_batch_qsc(ghz, allup, alldown, 2**5, 0.5, flip="Z")
states, labels, errors = next(data_generator)

In [32]:
labels

tensor([1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1,
        0, 1, 0, 1, 1, 1, 1, 0])

In [33]:
qmps_ghz.predict(states)[0][labels == 0]

tensor([[0.9551, 0.0449],
        [0.0016, 0.9984],
        [0.9551, 0.0449],
        [0.9551, 0.0449],
        [0.0016, 0.9984],
        [0.9551, 0.0449],
        [0.0016, 0.9984],
        [0.0016, 0.9984],
        [0.0016, 0.9984],
        [0.9551, 0.0449],
        [0.9551, 0.0449],
        [0.9551, 0.0449],
        [0.9551, 0.0449],
        [0.0016, 0.9984],
        [0.0016, 0.9984],
        [0.9551, 0.0449]], dtype=torch.float64, grad_fn=<IndexBackward0>)